# Demo — Inferencia con los 4 modelos (CarDD)

Notebook liviano para demostración: se elige una imagen de una carpeta, se corre inferencia con
los 4 modelos ya entrenados de la tesis (YOLOv8-det, Faster R-CNN, YOLOv8-seg, Mask R-CNN) y se
muestran las predicciones (detección/segmentación + clase de daño) junto con las métricas de
evaluación ya calculadas para cada modelo sobre el conjunto de test completo.

Este notebook **no reentrena ni recalcula métricas**: solo carga los pesos ya persistidos en
`artifacts/models/` y lee las tablas de métricas ya generadas en `artifacts/metrics/`. Ejecutar
las celdas en orden (kernel `.venv` del proyecto).

In [1]:
import json
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "artifacts" / "models" / "model_registry.json").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODELS_DIR = ARTIFACTS_DIR / "models"
METRICS_DIR = ARTIFACTS_DIR / "metrics"

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from notebook_training_helpers import build_faster_rcnn_model, build_mask_rcnn_model, pil_to_float_tensor

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")

PROJECT_ROOT: c:\proyectos python\Tesis 2026\Proyecto
Device: cuda:0


## 1. Configuración y clases de daño

Se lee `model_registry.json` (rutas de pesos por modelo) y `rcnn_arch_config.json` (arquitectura mínima necesaria para reconstruir Faster R-CNN / Mask R-CNN antes de cargar sus pesos).

In [2]:
MODEL_REGISTRY = json.loads((MODELS_DIR / "model_registry.json").read_text(encoding="utf-8"))
RCNN_ARCH_CONFIG = json.loads((MODELS_DIR / "rcnn_arch_config.json").read_text(encoding="utf-8"))

MODELS_BY_ID = {m["id"]: m for m in MODEL_REGISTRY["models"]}
CATEGORY_ID_TO_NAME = {int(k): v for k, v in MODEL_REGISTRY["classes"].items()}
NUM_TORCHVISION_CLASSES = len(CATEGORY_ID_TO_NAME) + 1  # +1 por la clase de fondo
TORCH_LABEL_TO_CATEGORY_ID = {i: cid for i, cid in enumerate(sorted(CATEGORY_ID_TO_NAME), start=1)}

_VIZ_PALETTE = [
    (231, 76, 60), (52, 152, 219), (46, 204, 113),
    (241, 196, 15), (155, 89, 182), (230, 126, 34),
]
VIZ_COLORS = {cid: _VIZ_PALETTE[i % len(_VIZ_PALETTE)] for i, cid in enumerate(sorted(CATEGORY_ID_TO_NAME))}

MODEL_ORDER = ["yolov8_det", "faster_rcnn", "yolov8_seg", "mask_rcnn"]

pd.DataFrame(MODEL_REGISTRY["models"])[["id", "display_name", "task", "backend", "weight_file"]]

,id,display_name,task,backend,weight_file
0,yolov8_det,YOLOv8n,detection,ultralytics,yolov8_det.pt
1,faster_rcnn,Faster R-CNN,detection,torchvision,faster_rcnn.pth
2,yolov8_seg,YOLOv8n-seg,segmentation,ultralytics,yolov8_seg.pt
3,mask_rcnn,Mask R-CNN,segmentation,torchvision,mask_rcnn.pth


## 2. Carga de modelos (perezosa y cacheada)

Cada modelo se instancia una sola vez por sesión de kernel. Faster R-CNN / Mask R-CNN se reconstruyen con `build_faster_rcnn_model`/`build_mask_rcnn_model` (mismas funciones que usa el notebook de entrenamiento) y luego cargan su `state_dict` entrenado.

In [3]:
_MODEL_CACHE: dict = {}


def load_demo_model(model_id: str):
    '''Carga (y cachea) un modelo por id: 'yolov8_det', 'faster_rcnn', 'yolov8_seg' o 'mask_rcnn'.'''
    if model_id in _MODEL_CACHE:
        return _MODEL_CACHE[model_id]

    spec = MODELS_BY_ID[model_id]
    weight_path = spec["weight_path"]

    if spec["backend"] == "ultralytics":
        from ultralytics import YOLO
        model = YOLO(weight_path)
    elif model_id in ("faster_rcnn", "mask_rcnn"):
        builder = build_faster_rcnn_model if model_id == "faster_rcnn" else build_mask_rcnn_model
        config = RCNN_ARCH_CONFIG[model_id]
        model = builder(config, {"num_torchvision_classes": NUM_TORCHVISION_CLASSES})
        checkpoint = torch.load(weight_path, map_location="cpu")
        model.load_state_dict(checkpoint["model_state_dict"])
        model = model.to(DEVICE).eval()
    else:
        raise KeyError(f"Modelo desconocido: {model_id}")

    _MODEL_CACHE[model_id] = model
    return model

## 3. Inferencia sobre una imagen

Cada backend (Ultralytics vs. torchvision) tiene su propia forma de preprocesar y devolver resultados; se normalizan a una lista común de dicts `{category_id, score, bbox, mask}`.

In [4]:
def _yolo_result_to_records(result, task):
    records = []
    boxes = result.boxes
    if boxes is None:
        return records

    xyxy = boxes.xyxy.cpu().numpy()
    conf = boxes.conf.cpu().numpy()
    cls = boxes.cls.cpu().numpy().astype(int)
    for i in range(len(xyxy)):
        records.append({
            "category_id": int(cls[i]) + 1,  # YOLO usa clases 0-indexadas; category_id es 1-indexado
            "score": float(conf[i]),
            "bbox": xyxy[i].tolist(),
            "mask": None,
        })

    if task == "segmentation" and result.masks is not None:
        h, w = result.orig_shape
        mask_data = result.masks.data.cpu().numpy()
        for i, rec in enumerate(records):
            resized = cv2.resize(mask_data[i], (w, h), interpolation=cv2.INTER_LINEAR)
            rec["mask"] = resized >= 0.5

    return records


def _torchvision_output_to_records(output, task, conf_threshold):
    boxes = output["boxes"].detach().cpu().numpy()
    scores = output["scores"].detach().cpu().numpy()
    labels = output["labels"].detach().cpu().numpy()
    masks = output.get("masks")

    records = []
    for i in range(len(boxes)):
        if scores[i] < conf_threshold:
            continue
        category_id = TORCH_LABEL_TO_CATEGORY_ID.get(int(labels[i]))
        if category_id is None:
            continue
        record = {
            "category_id": category_id,
            "score": float(scores[i]),
            "bbox": boxes[i].tolist(),
            "mask": None,
        }
        if task == "segmentation" and masks is not None:
            record["mask"] = masks[i, 0].detach().cpu().numpy() >= 0.5
        records.append(record)

    return records


def predict_single_image(model_id: str, image_path: Path, conf_threshold: float = 0.4):
    '''Corre inferencia de un modelo sobre una imagen; devuelve una lista de dicts
    {category_id, score, bbox, mask} -- mismo formato para los 4 backends.'''
    spec = MODELS_BY_ID[model_id]
    model = load_demo_model(model_id)

    if spec["backend"] == "ultralytics":
        result = model.predict(
            source=str(image_path), verbose=False, conf=conf_threshold,
            device=0 if DEVICE.type == "cuda" else "cpu",
        )[0]
        return _yolo_result_to_records(result, spec["task"])

    image = Image.open(image_path).convert("RGB")
    image_tensor = pil_to_float_tensor(image).to(DEVICE)
    with torch.no_grad():
        output = model([image_tensor])[0]
    return _torchvision_output_to_records(output, spec["task"], conf_threshold)

## 4. Visualización — grid 2x2

Cajas de color por clase de daño (+ score) y, para los modelos de segmentación, máscara semitransparente. Panel superior = detección (YOLOv8-det, Faster R-CNN), panel inferior = segmentación (YOLOv8-seg, Mask R-CNN).

In [5]:
def render_prediction_image(image_path: Path, pred_recs: list) -> Image.Image:
    '''Dibuja cajas de color por clase (+ score) y, si hay máscaras, las superpone semitransparentes.'''
    img = Image.open(image_path).convert("RGB")

    if any(rec["mask"] is not None for rec in pred_recs):
        overlay = img.convert("RGBA")
        for rec in pred_recs:
            if rec["mask"] is None:
                continue
            r, g, b = VIZ_COLORS.get(rec["category_id"], (180, 180, 180))
            mask = rec["mask"]
            mask_rgba = np.zeros((*mask.shape, 4), dtype=np.uint8)
            mask_rgba[mask] = [r, g, b, 100]
            overlay = Image.alpha_composite(overlay, Image.fromarray(mask_rgba, "RGBA"))
        img = overlay.convert("RGB")

    draw = ImageDraw.Draw(img)
    for rec in pred_recs:
        x1, y1, x2, y2 = (int(v) for v in rec["bbox"])
        r, g, b = VIZ_COLORS.get(rec["category_id"], (180, 180, 180))
        color = f"#{r:02x}{g:02x}{b:02x}"
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        label = f"{CATEGORY_ID_TO_NAME.get(rec['category_id'], '?')} {rec['score']:.2f}"
        draw.text((x1 + 2, max(0, y1 - 14)), label, fill=color)

    return img


def run_demo(image_path: Path, conf_threshold: float = 0.4):
    '''Corre los 4 modelos sobre `image_path`, muestra un grid 2x2 y un resumen de las clases
    de daño que reconoce cada modelo (con cuántas instancias de cada una).'''
    fig, axes = plt.subplots(2, 2, figsize=(13, 11))
    summary_rows = []

    for ax, model_id in zip(axes.flat, MODEL_ORDER):
        spec = MODELS_BY_ID[model_id]
        pred_recs = predict_single_image(model_id, image_path, conf_threshold=conf_threshold)
        rendered = render_prediction_image(image_path, pred_recs)

        ax.imshow(rendered)
        ax.set_title(f"{spec['display_name']} ({spec['task']}) -- {len(pred_recs)} instancia(s)")
        ax.axis("off")

        class_counts: dict = {}
        for rec in pred_recs:
            clase = CATEGORY_ID_TO_NAME.get(rec["category_id"], "?")
            class_counts[clase] = class_counts.get(clase, 0) + 1
        clases_str = ", ".join(f"{clase} x{n}" for clase, n in sorted(class_counts.items())) or "sin detecciones"
        summary_rows.append({
            "modelo": spec["display_name"],
            "clases_detectadas": clases_str,
            "total_instancias": len(pred_recs),
        })

    fig.tight_layout()
    plt.show()

    display(pd.DataFrame(summary_rows))

## 5. Seleccionar imagen y ejecutar

Escribe la ruta de una carpeta con imágenes (por defecto, el test set crudo de CarDD) y elegí un archivo del desplegable. Cualquier carpeta con `.jpg`/`.png` funciona, no hace falta que tenga anotaciones. Ajustá el umbral de confianza si querés ver más/menos detecciones y presioná **Ejecutar inferencia**.

In [ ]:
DEFAULT_IMAGE_DIR = PROJECT_ROOT / "data" / "raw" / "CarDD" / "CarDD_COCO" / "test2017"

folder_text = widgets.Text(
    value=str(DEFAULT_IMAGE_DIR),
    description="Carpeta:",
    layout=widgets.Layout(width="600px"),
    style={"description_width": "initial"},
)
image_dropdown = widgets.Dropdown(description="Imagen:", layout=widgets.Layout(width="400px"))
conf_slider = widgets.FloatSlider(
    value=0.4, min=0.05, max=0.9, step=0.05, description="Confianza mínima:",
    style={"description_width": "initial"},
)
run_button = widgets.Button(description="Ejecutar inferencia", button_style="primary", icon="play")
output_area = widgets.Output()


def _refresh_image_options(*_):
    folder = Path(folder_text.value)
    if not folder.is_dir():
        image_dropdown.options = []
        return
    files = sorted(
        p.name for p in folder.iterdir()
        if p.suffix.lower() in (".jpg", ".jpeg", ".png")
    )
    image_dropdown.options = files


def _on_run_clicked(_):
    output_area.clear_output(wait=True)
    with output_area:
        if not image_dropdown.value:
            print("Elegí una imagen del desplegable.")
            return
        image_path = Path(folder_text.value) / image_dropdown.value
        print(f"Imagen: {image_path}")
        display(Image.open(image_path).convert("RGB").copy())
        run_demo(image_path, conf_threshold=conf_slider.value)


folder_text.observe(_refresh_image_options, names="value")
run_button.on_click(_on_run_clicked)
_refresh_image_options()

display(widgets.VBox([
    widgets.HBox([folder_text]),
    widgets.HBox([image_dropdown, conf_slider, run_button]),
    output_area,
]))

# c:\proyectos python\Tesis 2026\Proyecto\data\raw\CarDD\CarDD_COCO\test2017
# C:\proyectos python\Tesis 2026\Proyecto\data\raw\CarDD\CarDD_COCO\demo

: 